# LangGraph Practice # 5


### 구성도

![구성도]()

In [1]:
import os,sys
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('utils'), '..')))
from module.utils import * 
from module.prompt import * 
from module.custom_model import *
from typing import List

In [2]:
start_langsmith('practive_5')

LangSmith 추적을 시작합니다.
[프로젝트명]
practive_5


In [3]:
from typing import TypedDict, Annotated, List, Literal
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage,ToolMessage

In [4]:
class GradeDocument(BaseModel):
    """
        you can says 'yes' or 'no'
        If this document is relevance about question, you say 'yes' and otherwise 'no'
    """
    answer:Literal['yes','no'] = Field(
        ...,
        description="Documents are relevant to the question, 'yes' or 'no'"
        
    )

loader = get_pdf_loader()
splitter = get_text_splitter()
docs = get_docs(loader,splitter)
embedding = get_embedding()
retriever = get_retriever(docs,embedding,k=7)



In [5]:
class State(TypedDict):
    question : Annotated[str,'user input question']   # 사용자 질의 or requeustion 질의
    messages : Annotated[list,add_messages]  # llm , retriever 등에서 생성된 데이터 원본 
    answer : Annotated[str,'llm final answer'] # llm 이 생성한 최종 대답
    documents : Annotated[list,'use retriever'] # retreiver 정제한 데이터

In [6]:
def retrieve(state : State)-> State:
    """ 
        사용자의 question을 받아 vectorestore에서 retriever 하는 과정
        3개의 chunk를 판단함
    """
    question = state['question']
    documents = retriever.invoke(question)
    messages = convert_docs_str(documents)
    return State({'messages':messages})

def grade_document(state : State)->State:
    """
        document의 관련성 여부를 판단하여 관련성 있는 파일만 추출하여 수집하는 단계
    """
    prompt = get_prompt_grade()
    llm = get_gemini(temperature=0.5)
    llm_with_grade=llm.with_structured_output(GradeDocument)
    chain = prompt | llm_with_grade
    origin_docs = convert_str_to_docs(state['messages'])
    filter_docs = []
    for doc in origin_docs:
        response = chain.invoke({'question':state['question'],'document':doc.page_content})
        if response.answer == 'yes':
            filter_docs.append(doc)
    return State({'documents':filter_docs})

def query_re_write(state : State)-> State:
    """
        grade_docuemtns 결과 docs 문서 수가 3이하인경우 `question` 변경하여 재요청
    """
    chain = get_prompt_re_write() | get_gemini(temperature=0.7) | StrOutputParser()
    new_question = chain.invoke({'question':state['question']})
    return State({'question':new_question})


def generate(state : State)->State:
    """
        grade_docuemtns 결과 docs 문서 수가 3이상인 경우 문서 생성
    """
    chain = get_prompt_rag() | get_gemini(temperature=0.7) | StrOutputParser()
    answer = chain.invoke({'question':state['question'],'context':state['documents']})
    return State({'answer':answer})



In [7]:
def is_grade(state : State)->Literal['generate','query_re_write']:
    if len(state['documents'])  >= 3:
        return 'generate'
    else: 
        return 'query_re_write'


def is_hallu_relevance(state : State)->Literal['generate','query_re_write',END]:
    chain =  get_relevant(llm=get_gemini(),target='retrieval-answer') 
    hallucination = chain.invoke({'answer':state['answer'],'context':state['documents']})
    if hallucination.score == 'yes' : # document 와 answer가 연관 있다
        _chain =  get_relevant(llm=get_gemini(),target='question-answer') 
        relevant = _chain.invoke({'question':state['question'],'answer':state['answer']})
        if relevant.score =='yes':
            return END
        else:
            return 'query_re_write'

    else : # document 와 answer가 연관 없다
        return 'generate'

In [8]:
state_graph = StateGraph(State)
state_graph.add_node('retrieve',retrieve)
state_graph.add_node('grade_document',grade_document)
state_graph.add_node('query_re_write',query_re_write)
state_graph.add_node('generate',generate)

state_graph.add_edge(START,'retrieve')
state_graph.add_edge('retrieve','grade_document')
state_graph.add_conditional_edges(
    source='grade_document',
    path=is_grade,
    path_map=['generate','query_re_write']
)
state_graph.add_conditional_edges(
    source='generate',
    path=is_hallu_relevance
)

state_graph.add_edge('query_re_write','retrieve')

memory = get_check_pointer()

graph = state_graph.compile(checkpointer=memory)



In [ ]:
# visualize_graph(graph)
print(graph.get_graph().draw_mermaid())

그래프 시각화 실패 (추가 종속성 필요): Failed to reach https://mermaid.ink/ API while trying to render your graph. Status code: 204.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`
ASCII로 그래프 표시:
ASCII 표시도 실패: no intersection found (point inside ?!). view: <langchain_core.runnables.graph_ascii.VertexViewer object at 0x755e587b10d0> topt: (-12.0, 19.5)


In [10]:
uuid = get_random_uuid()
config = get_runnable_config(recursion_limit=10,thread_id=uuid)

In [11]:
inputs ={'question':'오늘의 뉴스'}
stream_graph(graph,inputs,config)


🔄 Node: grade_document 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: query_re_write 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
오늘의 주요 뉴스 요약
🔄 Node: grade_document 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: generate 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
- 미국, 안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령 발표
- G7, 히로시마 AI 프로세스를 통해 AI 기업 대상 국제 행동강령에 합의
- 영국 AI 안전성 정상회의에 참가한 28개국, AI 위험에 공동 대응 선언
- 미국 법원, 예술가들이 생성 AI 기업에 제기한 저작권 소송 기각
- 미국 연방거래위원회, 저작권청에 소비자 보호와 경쟁 측면의 AI 의견서 제출
- EU AI 법 3자 협상, 기반모델 규제 관련 견해차로 난항
- 미국 프런티어 모델 포럼, 1,000만 달러 규모의 AI 안전 기금 조성
- 코히어, 데이터 투명성 확보를 위한 데이터 출처 탐색기 공개
- 알리바바 클라우드, 최신 LLM ‘통이치엔원 2.0’ 공개
- 삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
- 구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화
- IDC, 2027년 AI 소프트웨어 매출 2,500억 달러 돌파 전망
- 빌 게이츠, AI 에이전트로 인한 컴퓨터 사용의 패러다임 변화 전망
- 유튜브, 2024년부터 AI 생성 콘텐츠 표시 의무화
- 영국 과학혁신기술부, AI 안전 연구소 설립 발표
- 구글 딥마인드, 범용 AI 모델의 기능과 동작에 대한 분류 체계 발표
- 갈릴레오의 LLM 환각 지수 평가에서 GPT-4가 가장 우수
- 영국 옥스퍼드 인터넷 연구소, AI 기술자의 임금이 평균 2

In [12]:
invoke_graph(graph,inputs,config)


🔄 Node: retrieve 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
messages:
<document><content>Ⅰ. 인공지능 산업 동향 브리프</content><source>../data/SPRI_AI_Brief_2023년12월호_F.pdf</source><page>3</page></document>
<document><content>2023 년 12월호</content><source>../data/SPRI_AI_Brief_2023년12월호_F.pdf</source><page>1</page></document>
<document><content>2023 년 12월호
Ⅰ. 인공지능 산업 동향 브리프
 1. 정책/법제 
   ▹ 미국, 안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령 발표  ························· 1
   ▹ G7, 히로시마 AI 프로세스를 통해 AI 기업 대상 국제 행동강령에 합의··························· 2
   ▹ 영국 AI 안전성 정상회의에 참가한 28개국, AI 위험에 공동 대응 선언··························· 3
   ▹ 미국 법원, 예술가들이 생성 AI 기업에 제기한 저작권 소송 기각····································· 4
   ▹ 미국 연방거래위원회 , 저작권청에 소비자 보호와 경쟁 측면의 AI 의견서 제출················· 5
   ▹ EU AI 법 3자 협상, 기반모델 규제 관련 견해차로 난항··················································· 6
 
 2. 기업/산업 
   ▹ 미국 프런티어 모델 포럼, 1,000 만 달러 규모의 AI 안전 기금 조성································ 7
   ▹ 코히어 , 데이터 투명성 확보를 위한 데이터 출처 탐색기 공개  ···········